# Copycat

Copycat is a small concatenative language in which a model invocation is a
first-class program-synthesis effect. The deterministic runtime stays in
control: `{natural language}` asks a backend for Copycat code, and that code
continues immediately against the current data stack.

The language implementation lives in the installable `copycat` package. This
notebook contains its interactive tests and examples.


## Setup — run this section first

Run these cells in Google Colab to install Copycat from GitHub, import its public
API, and fetch the reusable data-structure module. The optional Gemma dependencies
are installed only in the live-model section below.


### Install the package


In [ ]:
%pip install -q "copycat-language[test] @ git+https://github.com/dearmadisonblue/copycat.git@main"


In [ ]:
from copycat import (
    Abstract,
    Annotation,
    Catenate,
    CopycatError,
    EvaluationError,
    GeneratedCodeError,
    Gemma4Backend,
    Model,
    ModelProtocolError,
    ModelReportedError,
    ModelTurn,
    Module,
    ModuleDocumentationWarning,
    ParseError,
    StubModel,
    read,
    run,
)


### Fetch the standard data module

`Module.load` automatically runs every `test-*` word in the archive, so this cell
also smoke-tests the module against the currently installed evaluator.


In [ ]:
from urllib.request import urlretrieve

DATA_MODULE_URL = (
    "https://raw.githubusercontent.com/dearmadisonblue/copycat/"
    "main/modules/data.module"
)
urlretrieve(DATA_MODULE_URL, "data.module")
data_module = Module.load("data.module")
print(f"Loaded {len(data_module)} words from data.module.")


## Tests


In [ ]:
import ipytest
import pytest

ipytest.autoconfig()


In [ ]:
%%ipytest -q


@pytest.mark.parametrize(
    "source, expected",
    [
        ("[foo] d", "[foo] [foo]"),
        ("[foo] e", ""),
        ("[foo] [bar] f", "[bar] [foo]"),
        ("[foo] [bar] c", "[foo bar]"),
        ("[foo] b", "[[foo]]"),
        ("[foo] a", "foo"),
        ("[foo] s bar qux r baz", "[bar qux] foo baz"),
    ],
)
def test_original_examples(source, expected):
    assert run(source, verbose=False) == expected


def test_shift_finds_reset_when_reset_is_the_final_instruction():
    assert run("[a] s 1 r", verbose=False) == "1"


def test_model_form_is_opaque_to_copycat_syntax():
    program = read('{write [this] and "that"\non two lines}')
    (model,) = program.body
    assert isinstance(model, Model)
    assert model.prompt == 'write [this] and "that"\non two lines'


def test_reader_uses_complete_syntax_class_names():
    program = read("[d]")
    (quotation,) = program.body

    assert isinstance(program, Catenate)
    assert isinstance(quotation, Abstract)


def test_annotations_parse_and_print():
    program = read("@eq @trace")
    assert all(isinstance(term, Annotation) for term in program.body)
    assert str(program) == "@eq @trace"


def test_unknown_annotation_is_identity():
    assert run("1 @trace", verbose=False) == "1"


def test_eq_annotation_is_identity_when_values_match():
    assert run("1 1 @eq", verbose=False) == "1 1"


def test_eq_annotation_fails_when_values_differ():
    with pytest.raises(EvaluationError, match="@eq assertion failed"):
        run("1 2 @eq", verbose=False)


@pytest.mark.parametrize(
    "source, fragment",
    [
        ("[d", "Unclosed quotation"),
        ("d]", "no matching '['"),
        ("{do something", "Unclosed model invocation"),
        ('"unterminated', "unterminated string"),
        ("d @", "Invalid annotation"),
        ("g", "Single-letter names are reserved"),
        ("Foo", "Unexpected character 'F'"),
        ("foo_bar", "Unexpected character '_'")
    ],
)
def test_parser_errors_are_explanatory(source, fragment):
    with pytest.raises(ParseError) as caught:
        read(source)
    assert fragment.lower() in str(caught.value).lower()


def test_hyphenated_user_word_names_parse():
    assert str(read("foo-doc long-word-2")) == "foo-doc long-word-2"


def test_stack_underflow_residualizes():
    assert run("d", verbose=False) == "d"


def test_run_forwards_gas_budget():
    assert run("1 d", gas=2, verbose=False) == "1 d"


def test_module_bodies_are_source_text_and_cached():
    module = Module({
        "duplicate": "d",
        "duplicate-doc": '"Duplicate the top value. Example: 1 duplicate ==> 1 1."',
    })

    assert run("1 duplicate", module=module, verbose=False) == "1 1"
    assert run("duplicate-doc", module=module, verbose=False) == (
        '"Duplicate the top value. Example: 1 duplicate ==> 1 1."'
    )
    assert module.parsed("duplicate") is module.parsed("duplicate")


def test_module_rejects_single_letter_user_words():
    with pytest.raises(ValueError, match="longer than one character"):
        Module({"g": "d"})


def test_module_round_trip(tmp_path):
    module = Module({
        "duplicate": "d",
        "duplicate-doc": '"Duplicate the top value. Example: 1 duplicate ==> 1 1."',
        "duplicate-twice": "d d",
        "duplicate-twice-doc": (
            '"Duplicate the top value twice. Example: 1 duplicate-twice ==> 1 1 1."'
        ),
    })
    path = tmp_path / "example.module"

    module.save(path)
    loaded = Module.load(path)

    assert dict(loaded) == dict(module)
    assert isinstance(loaded, Module)


def test_module_runs_smoke_tests_on_load(tmp_path):
    module = Module({
        "identity": "",
        "identity-doc": '"Leave the stack unchanged. Example: 1 identity ==> 1."',
        "test-identity": "1 identity 1 @eq",
    })
    path = tmp_path / "passing.module"
    module.save(path)

    assert dict(Module.load(path)) == dict(module)


def test_module_rejects_failed_smoke_test():
    with pytest.raises(EvaluationError, match="@eq assertion failed"):
        Module({"test-broken": "1 2 @eq"})


def test_repository_data_module_loaded_and_smoke_tested():
    assert run(
        "7 singleton head [0] [] option-case",
        module=data_module,
        verbose=False,
    ) == "7"


def test_module_is_immutable_and_copies_its_sources():
    sources = {
        "identity": "",
        "identity-doc": '"Leave the stack unchanged. Example: 1 identity ==> 1."',
    }
    module = Module(sources)
    sources["identity"] = "d"

    assert module["identity"] == ""
    with pytest.raises(TypeError):
        module["identity"] = "d"



def test_missing_documentation_warns_and_is_cataloged_as_undocumented():
    with pytest.warns(ModuleDocumentationWarning, match="undocumented"):
        module = Module({"identity": ""})

    assert str(module) == "identity\n  (undocumented)"



def test_non_string_documentation_warns_and_is_unavailable():
    with pytest.warns(ModuleDocumentationWarning, match="exactly one string"):
        module = Module({"identity": "", "identity-doc": "1"})

    assert module.documentation("identity") is None
    assert "(undocumented)" in str(module)



def test_syntax_error_in_documentation_rejects_module():
    with pytest.raises(ParseError, match="unterminated string"):
        Module({"identity": "", "identity-doc": '"unterminated'})



def test_module_string_has_only_word_names_and_documentation():
    module = Module({
        "duplicate": "d",
        "duplicate-doc": '"Duplicate the top value. Example: 1 duplicate ==> 1 1."',
        "test-duplicate": "1 duplicate 1 @eq",
    })

    assert str(module) == (
        "duplicate\n"
        "  Duplicate the top value. Example: 1 duplicate ==> 1 1."
    )
    assert "test-duplicate" not in str(module)
    assert "duplicate-doc" not in str(module)
    assert "\nd\n" not in str(module)



def test_model_effect_receives_cached_module_catalog():
    module = Module({
        "duplicate": "d",
        "duplicate-doc": '"Duplicate the top value. Example: 1 duplicate ==> 1 1."',
    })
    backend = StubModel("<OK>duplicate</OK>")

    assert run(
        "1 {duplicate it}",
        module=module,
        model_backend=backend,
        verbose=False,
    ) == "1 1"
    assert backend.calls == [("duplicate it", "1", str(module))]



def test_verbose_model_effect_reports_prompt_tokens(capsys):
    class CountingBackend:
        def generate(self, *, prompt, stack, module_catalog):
            return ModelTurn(answer="<OK></OK>", prompt_tokens=1234)

    run("{do nothing}", model_backend=CountingBackend())

    assert "Prompt tokens: 1,234" in capsys.readouterr().out


def test_stub_model_ok_executes_generated_code_immediately():
    backend = StubModel("<OK>f</OK>")

    assert run(
        "1 2 {swap the top two values}",
        model_backend=backend,
        verbose=False,
    ) == "2 1"

    assert backend.calls == [
        ("swap the top two values", "1 2", "(none)")
    ]


def test_stub_model_error_becomes_structured_condition():
    backend = StubModel("<ERROR>I cannot do that safely.</ERROR>")

    with pytest.raises(ModelReportedError):
        run(
            "1 {do something impossible}",
            model_backend=backend,
            verbose=False,
        )


def test_protocol_allows_surrounding_text_and_lowercase_tags():
    backend = StubModel("I chose this.\n<ok>d</ok>\nDone.")

    assert run(
        "1 {duplicate the value}",
        model_backend=backend,
        verbose=False,
    ) == "1 1"


def test_protocol_rejects_multiple_expected_elements():
    backend = StubModel("<OK>d</OK> or <ERROR>unsure</ERROR>")

    with pytest.raises(ModelProtocolError):
        run(
            "1 {do something impossible}",
            model_backend=backend,
            verbose=False,
        )


def test_bad_generated_syntax_is_attributed_to_model_output():
    backend = StubModel("<OK>[d</OK>")

    with pytest.raises(GeneratedCodeError) as caught:
        run(
            "1 {do it}",
            model_backend=backend,
            verbose=False,
        )

    assert "<model output>" in str(caught.value)


def test_generated_code_can_invoke_the_model():
    class NestedModel:
        def __init__(self):
            self.answers = iter(["<OK>{ask again}</OK>", "<OK>d</OK>"])
            self.prompts = []

        def generate(self, *, prompt, stack, module_catalog):
            self.prompts.append(prompt)
            return ModelTurn(answer=next(self.answers))

    backend = NestedModel()

    assert run(
        "1 {do it}",
        model_backend=backend,
        verbose=False,
    ) == "1 1"
    assert backend.prompts == ["do it", "ask again"]


## Examples


Run this section in Colab to exercise the reader, modules, deterministic
model effects, then load Gemma and run the live examples. Evaluator tracing is
intentionally enabled in the executable examples.


### Reader examples


In [ ]:
for source in [
    "1 2 f",
    "[foo] s bar qux r baz",
    "foo-doc",
    "1 1 @eq",
    '{write [this] and "that" on two lines}',
]:
    print(f"{source!r} -> {read(source)!r}")


### Module examples


In [ ]:
module = Module({
    "duplicate": "d",
    "duplicate-doc": (
        '"duplicate (x -- x x): Duplicate the top value. '
        'Example: 1 duplicate ==> 1 1."'
    ),
})

print(run("1 duplicate", module=module))
print(run("duplicate-doc", module=module))

module.save("example.module")
loaded_module = Module.load("example.module")
print(loaded_module)

print(run("1 2 pair first", module=data_module))
print(run("7 singleton head [0] [] option-case", module=data_module))


### Deterministic model examples


In [ ]:
swap_stub = StubModel("Reasoning outside the tag is accepted.\n<ok>f</ok>")
copy_stub = StubModel("<OK>d</OK>")

print(
    run(
        "1 2 {put the top two values in the opposite order}",
        model_backend=swap_stub,
    )
)

print(
    run(
        '"hello" {duplicate the top value}',
        model_backend=copy_stub,
    )
)


### Live Gemma 4 synthesis


#### Load Gemma


In [ ]:
%pip install -q -U "copycat-language[gemma] @ git+https://github.com/dearmadisonblue/copycat.git@main"


In [ ]:
# Optional, only if your Hugging Face environment asks for authentication:
# from huggingface_hub import notebook_login
# notebook_login()

gemma = Gemma4Backend.load(
    max_new_tokens=8_192,
    stream_output=True,
)


#### Run live examples


In [ ]:
examples = [
    "1 2 {put the top two values in the opposite order}",
    '"hello" {duplicate the top value}',
    "{put the number 7 on the stack}",
]

for source in examples:
    print("\n" + "=" * 72)
    print("SOURCE:", source)
    try:
        print(
            "RESULT:",
            run(
                source,
                model_backend=gemma,
            ),
        )
    except CopycatError as exc:
        print(exc)


## Future work

This version remains deliberately narrow: one model turn synthesizes a small
Copycat program, that program is parsed, and ordinary evaluation continues.
Module reflection is deliberately deferred until the effect and handler
model is settled. Repair loops, generic effect handlers, capabilities, external
services, simulation, persisted continuations, and actors also remain future work.
